# 🎹 Piano Waterfall — Free AI OMR Server (Google Colab)

Chạy server AI OMR (`homr`) **miễn phí 100%** trên Google Colab để chuyển đổi bản nhạc **PDF nhiều trang → MusicXML chuẩn** và tự động kết nối với web app [Piano Waterfall](https://elihasad.github.io/piano-waterfall/).

### Cách dùng:
1. Bấm nút **▶ (Play)** ở ô bên dưới.
2. Chờ khoảng **1–2 phút** để cài `homr` và mở link HTTPS.
3. Khi hiện nút **🎹 Mở Piano Waterfall (Đã tự gắn AI OMR)**, bấm vào là dùng được ngay trên web!

In [ ]:
#@title 🎹 Bấm nút Play (▶) ở bên trái để khởi động AI OMR Server
import os, re, shutil, subprocess, sys, time, urllib.request
from urllib.parse import quote
from IPython.display import HTML, display

REPO_DIR = '/content/piano-waterfall'
BACKEND_DIR = f'{REPO_DIR}/services/omr-backend'
VENV_DIR = '/content/omr_venv'

def run_cmd(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.returncode != 0:
        print(p.stdout[-3000:])
        raise RuntimeError(f'Lệnh thất bại ({p.returncode}): {" ".join(cmd)}')
    return p.stdout

print('1/4 📥 Đang tải mã nguồn Piano Waterfall OMR Backend...', flush=True)
if not os.path.exists(REPO_DIR):
    run_cmd(['git', 'clone', '--depth', '1', 'https://github.com/Elihasad/piano-waterfall.git', REPO_DIR])
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

print('2/4 📦 Đang cài đặt môi trường AI OMR (homr + FastAPI)... (mất ~30 giây lần đầu)', flush=True)
venv_py = f'{VENV_DIR}/bin/python'
if not os.path.exists(venv_py):
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    run_cmd(['uv', 'venv', VENV_DIR])
run_cmd(['uv', 'pip', 'install', '--python', venv_py, 'fastapi>=0.115.0', 'uvicorn>=0.32.0', 'python-multipart>=0.0.12', 'pypdfium2>=4.30.0', 'pillow>=10.0.0', 'homr[cpu]>=0.3.2'])
print('    ✓ Đã cài xong homr:', f'{VENV_DIR}/bin/homr', 'tồn tại =', os.path.exists(f'{VENV_DIR}/bin/homr'), flush=True)

print('3/4 🌐 Đang tải Cloudflare Tunnel (miễn phí)...', flush=True)
cf_bin = '/usr/local/bin/cloudflared'
if not os.path.exists(cf_bin):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cf_bin,
    )
    os.chmod(cf_bin, 0o755)

print('4/4 🚀 Đang khởi động AI OMR Server...', flush=True)
env = os.environ.copy()
env['PATH'] = f'{VENV_DIR}/bin:' + env.get('PATH', '')

uvicorn_proc = subprocess.Popen(
    [venv_py, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '7860'],
    cwd=BACKEND_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

ready = False
for _ in range(30):
    if uvicorn_proc.poll() is not None:
        print(uvicorn_proc.stdout.read())
        raise RuntimeError('Server uvicorn bị dừng sớm.')
    try:
        with urllib.request.urlopen('http://127.0.0.1:7860/health', timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    uvicorn_proc.terminate()
    raise RuntimeError('Server uvicorn không phản hồi /health sau 30 giây.')

tunnel_proc = subprocess.Popen(
    [cf_bin, 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
url_pattern = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
for line in iter(tunnel_proc.stdout.readline, ''):
    m = url_pattern.search(line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    tunnel_proc.terminate()
    uvicorn_proc.terminate()
    raise RuntimeError('Không lấy được link Cloudflare Tunnel.')

magic_link = f'https://elihasad.github.io/piano-waterfall/?omr={quote(public_url, safe="")}'

display(HTML(f'''
<div style="padding:16px 20px;border-radius:14px;background:#1a1b26;color:#f8fafc;border:1px solid #7c3aed;font-family:system-ui,sans-serif;max-width:680px;margin-top:8px">
  <div style="font-size:16px;font-weight:700;margin-bottom:8px">✅ AI OMR Server đã sẵn sàng!</div>
  <div style="font-size:13px;color:#cbd5e1;margin-bottom:14px">Giữ nguyên tab Colab này đang chạy trong lúc tải file PDF lên Piano Waterfall.</div>
  <a href="{magic_link}" target="_blank" style="display:inline-block;padding:10px 18px;border-radius:10px;background:#7c3aed;color:#ffffff;font-weight:700;text-decoration:none;font-size:14px">
    🎹 Mở Piano Waterfall (Đã tự gắn AI OMR)
  </a>
  <div style="margin-top:12px;font-size:12px;color:#94a3b8">
    Link AI OMR (nếu cần dán lên điện thoại): <code style="color:#38bdf8;background:#0f172a;padding:3px 6px;border-radius:6px">{public_url}</code>
  </div>
</div>
'''))

print(f'AI OMR URL: {public_url}', flush=True)
print('Đang chạy server... (Bấm nút Stop vuông khi không dùng nữa)', flush=True)

try:
    for line in iter(uvicorn_proc.stdout.readline, ''):
        if line.strip():
            print(line.strip(), flush=True)
except KeyboardInterrupt:
    print('Đang tắt server...')
finally:
    tunnel_proc.terminate()
    uvicorn_proc.terminate()
